# Báo cáo giải đấu arena

Đọc log JSONL do `arena.tournament` sinh ra, tạo bảng và hình cho chương 4.

Đổi `LOG_PATH` thành log cần phân tích, rồi **Run All**. Hình được lưu vào `analysis/figures/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analysis.arena_stats import (agents_in, bootstrap_elo, color_table, error_games,
                                  load_games, pair_table, timing_table)

LOG_PATH = ROOT / 'logs' / 'smoke.jsonl'   # đổi thành log cần phân tích
N_BOOT = 1000
FIG_DIR = ROOT / 'analysis' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

games = load_games(LOG_PATH)
labels = agents_in(games)
print(f'{len(games)} ván, {len(error_games(games))} ván lỗi, tác tử: {labels}')

## 1. Ma trận đối đầu

Score của hàng gặp cột (thắng = 1, hòa = 0,5), kèm khoảng tin cậy Wilson 95%.

In [ ]:
pt = pair_table(games)
score = pd.DataFrame(np.nan, index=labels, columns=labels)
text = pd.DataFrame('', index=labels, columns=labels)
for (a, b), row in pt.items():
    lo, hi = row['ci95']
    score.loc[a, b] = row['score']
    text.loc[a, b] = f"{row['score']:.2f}\n[{lo:.2f}, {hi:.2f}]"

fig, ax = plt.subplots(figsize=(1.6 * len(labels) + 2, 1.3 * len(labels) + 1.5))
im = ax.imshow(score.values.astype(float), cmap='RdBu', vmin=0, vmax=1)
ax.set_xticks(range(len(labels)), labels, rotation=30, ha='right')
ax.set_yticks(range(len(labels)), labels)
for i, a in enumerate(labels):
    for j, b in enumerate(labels):
        v = score.loc[a, b]
        color = 'white' if (not np.isnan(v) and abs(v - 0.5) > 0.3) else 'black'
        ax.text(j, i, text.loc[a, b] or '–', ha='center', va='center', fontsize=9, color=color)
ax.set_xlabel('đối thủ (cột)')
ax.set_ylabel('tác tử (hàng)')
ax.set_title('Score của hàng gặp cột (Wilson 95%)')
fig.colorbar(im, ax=ax, fraction=0.046)
fig.tight_layout()
fig.savefig(FIG_DIR / f'{LOG_PATH.stem}_matrix.png', dpi=150)
plt.show()

In [ ]:
pairs = pd.DataFrame([
    {'A': a, 'B': b, 'n': r['n'], 'thắng': r['wins'], 'hòa': r['draws'], 'thua': r['losses'],
     'score A': round(r['score'], 3), 'Wilson 95%': f"[{r['ci95'][0]:.3f}, {r['ci95'][1]:.3f}]"}
    for (a, b), r in pt.items() if labels.index(a) < labels.index(b)
])
pairs

## 2. Xếp hạng Elo (Bradley-Terry)

Khoảng tin cậy 95% bằng bootstrap theo ván.

In [ ]:
elo = bootstrap_elo(games, n_boot=N_BOOT)
elo_df = (pd.DataFrame([{'tác tử': a, 'Elo': v['elo'], 'thấp': v['ci95'][0], 'cao': v['ci95'][1]}
                        for a, v in elo.items()])
          .sort_values('Elo', ascending=False).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(7, 0.6 * len(labels) + 1.5))
y = np.arange(len(elo_df))
err = [elo_df['Elo'] - elo_df['thấp'], elo_df['cao'] - elo_df['Elo']]
ax.barh(y, elo_df['Elo'], xerr=err, color='#4C78A8', capsize=4)
ax.set_yticks(y, elo_df['tác tử'])
ax.invert_yaxis()
ax.axvline(0, color='grey', lw=0.8)
ax.set_xlabel('Elo (trung bình = 0)')
ax.set_title(f'Bradley-Terry, bootstrap {N_BOOT} lần')
fig.tight_layout()
fig.savefig(FIG_DIR / f'{LOG_PATH.stem}_elo.png', dpi=150)
plt.show()
elo_df.round(0)

## 3. Lợi thế đi trước

Score khi cầm X (đi trước) và khi cầm O.

In [ ]:
ct = color_table(games)
print(f"Score chung của bên X: {ct['x_score']:.3f} [{ct['x_ci95'][0]:.3f}, {ct['x_ci95'][1]:.3f}]")
color_df = pd.DataFrame(ct['agents']).T[['score_as_x', 'score_as_o', 'games_as_x', 'games_as_o']]
ax = color_df[['score_as_x', 'score_as_o']].astype(float).plot.bar(figsize=(7, 3.5), rot=0)
ax.set_ylim(0, 1)
ax.set_ylabel('score')
ax.legend(['cầm X (đi trước)', 'cầm O'])
ax.set_title('Score theo màu quân')
plt.tight_layout()
plt.savefig(FIG_DIR / f'{LOG_PATH.stem}_color.png', dpi=150)
plt.show()
color_df

## 4. Thời gian và chi phí tìm kiếm

In [ ]:
tt = pd.DataFrame(timing_table(games)).T
times = {a: [] for a in labels}
for g in games:
    side = {1: g['agent_x']['label'], -1: g['agent_o']['label']}
    for mv in g['moves']:
        times[side[mv['player']]].append(mv['time_ms'])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.boxplot([times[a] for a in labels], showfliers=False)
ax.set_xticks(range(1, len(labels) + 1), labels)
ax.set_ylabel('ms / nước')
ax.set_title('Thời gian suy nghĩ mỗi nước')
fig.tight_layout()
fig.savefig(FIG_DIR / f'{LOG_PATH.stem}_time.png', dpi=150)
plt.show()
tt